In [1]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [llama_stack_client]lama_stack_client]


In [2]:
import os
import sys
from dotenv import load_dotenv
from llama_stack_client import LlamaStackClient
import pandas as pd
import logging
import requests
from io import BytesIO

In [26]:
sys.path.append('..')
# Load environment variables from .env file
load_dotenv()

logger = logging.getLogger(__name__)
logger.setLevel("INFO")

# Initialize the Llama Stack client
client = LlamaStackClient(
    base_url=os.getenv("LLAMA_STACK_SERVER_URL", "http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321")
)

file_path = "data/Commercial-Direct-LATAM-USD-Q3-2025-Subscriptions.csv"
url = "https://www.openshift.guide/openshift-guide-screen.pdf"
vector_db_skus_name = "skus_rh_vector_db"
vector_db_ocp_name = "ocp_rh_vector_db"

logger.info("Connected to Llama Stack server")

INFO:__main__:Connected to Llama Stack server


In [4]:
for m in client.models.list():
    print(f"Model: {m}")
    #if m.id == "sentence-transformers/nomic-ai/nomic-embed-text-v1.5":
        #client.models.unregister(
        #    model_id=m.id
        #)

INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/models "HTTP/1.1 200 OK"


Model: Model(id='sentence-transformers/ibm-granite/granite-embedding-125m-english', created=1777585730, owned_by='llama_stack', custom_metadata={'model_type': 'embedding', 'provider_id': 'sentence-transformers', 'provider_resource_id': 'ibm-granite/granite-embedding-125m-english', 'embedding_dimension': 768}, object='model')
Model: Model(id='vllm-inference/redhataillama-31-8b-instruct', created=1777585730, owned_by='llama_stack', custom_metadata={'model_type': 'llm', 'provider_id': 'vllm-inference', 'provider_resource_id': 'redhataillama-31-8b-instruct'}, object='model')
Model: Model(id='sentence-transformers/nomic-ai/nomic-embed-text-v1.5', created=1777585730, owned_by='llama_stack', custom_metadata={'model_type': 'embedding', 'provider_id': 'sentence-transformers', 'provider_resource_id': 'nomic-ai/nomic-embed-text-v1.5', 'embedding_dimension': 768}, object='model')


In [27]:
vector_stores = client.vector_stores.list()

print(f"Vector stores {vector_stores}")

for vector_store in vector_stores:
    client.vector_stores.delete(
        vector_store_id=vector_store.id
    )
    print(f"Vector store: {vector_store.name} deleted")

print("All Vector stores deleted")

vector_stores = client.vector_stores.list()

print(f"Vector stores {vector_stores}")

INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: DELETE http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores/vs_36e32757-5c9a-4f78-b6af-ea51441927cd "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: DELETE http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores/vs_3a033334-4294-4d2a-a2e8-5c65afaf9fd5 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Vector stores SyncOpenAICursorPage[VectorStore](data=[VectorStore(id='vs_36e32757-5c9a-4f78-b6af-ea51441927cd', created_at=1777585954, file_counts=FileCounts(cancelled=0, completed=3, failed=1, in_progress=0, total=4), expires_after=None, expires_at=None, last_active_at=1777585954, metadata={'provider_id': 'milvus', 'provider_vector_store_id': 'vs_36e32757-5c9a-4f78-b6af-ea51441927cd', 'embedding_model': 'sentence-transformers/ibm-granite/granite-embedding-125m-english', 'embedding_dimension': '768'}, name='ocp_rh_vector_db', object='vector_store', status='completed', usage_bytes=0), VectorStore(id='vs_3a033334-4294-4d2a-a2e8-5c65afaf9fd5', created_at=1777585744, file_counts=FileCounts(cancelled=0, completed=3, failed=2, in_progress=0, total=5), expires_after=None, expires_at=None, last_active_at=1777585744, metadata={'provider_id': 'milvus', 'provider_vector_store_id': 'vs_3a033334-4294-4d2a-a2e8-5c65afaf9fd5', 'embedding_model': 'sentence-transformers/ibm-granite/granite-embedding-12

In [28]:
# Create a vector store skus and index the file
vector_store_skus = client.vector_stores.create(
    name=vector_db_skus_name,
    extra_body={
        "provider_id": "milvus",
        "embedding_model": "sentence-transformers/ibm-granite/granite-embedding-125m-english",
        "embedding_dimension": 768,
    },
)

# Create a vector store ocp and index the file
vector_store_ocp = client.vector_stores.create(
    name=vector_db_ocp_name,
    extra_body={
        "provider_id": "milvus",
        "embedding_model": "sentence-transformers/ibm-granite/granite-embedding-125m-english",
        "embedding_dimension": 768,
    },
)

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


In [29]:
vector_db_skus_id = ""
vector_dbs = client.vector_stores.list()
for vector_db in vector_dbs:
    if vector_db.name == vector_db_skus_name:
        vector_db_skus_id = vector_db.id
        break
if vector_db_skus_id == "":
    print(f"Vector DB ID for SKUs: {vector_db_skus_name} not found in the vector stores")

print(f"Vector DB ID for SKUs: {vector_db_skus_id}")

INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Vector DB ID for SKUs: vs_2aba51eb-3c44-4d17-926e-136d358f8efa


In [30]:
# Upload a document
file = client.files.create(
    file=open(file_path, "rb"),
    purpose="assistants",
)
print(f"Uploaded: {file.id}")

client.vector_stores.files.create(
    vector_store_id=vector_db_skus_id,
    file_id=file.id,
    attributes={
        "document_id": "Subscriptions.csv",
        "source": file_path
    },
    chunking_strategy={
        "type": "static",
        "static": {"max_chunk_size_tokens": 512, "chunk_overlap_tokens": 128},
    },
)

print(f"File {file.id} loaded into Vector store SKUs with ID: {vector_db_skus_id}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/files "HTTP/1.1 200 OK"


Uploaded: file-8bfb4a13f53d4522850a0a66db3d69d8


INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores/vs_2aba51eb-3c44-4d17-926e-136d358f8efa/files "HTTP/1.1 200 OK"


File file-8bfb4a13f53d4522850a0a66db3d69d8 loaded into Vector store SKUs with ID: vs_2aba51eb-3c44-4d17-926e-136d358f8efa


In [31]:
vector_db_ocp_id = ""
vector_dbs = client.vector_stores.list()
for vector_db in vector_dbs:
    if vector_db.name == vector_db_ocp_name:
        vector_db_ocp_id = vector_db.id
        break
if vector_db_skus_id == "":
    print(f"Vector DB ID for SKUs: {vector_db_ocp_name} not found in the vector stores")

print(f"Vector DB ID for SKUs: {vector_db_ocp_id}")

INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Vector DB ID for SKUs: vs_35b4d3d7-6e9d-4adc-8d3f-686d9f62477b


In [32]:
response = requests.get(url)
file_buffer = BytesIO(response.content)
file_buffer.name = "openshift-guide-screen.pdf"

# Upload a document
file_url = client.files.create(
    file=file_buffer,
    purpose="assistants",
)
print(f"Uploaded: {file_url.id}")

client.vector_stores.files.create(
    vector_store_id=vector_db_ocp_id,
    file_id=file_url.id,
    attributes={
        "document_id": file_buffer.name,
        "source": url
    },
    chunking_strategy={
        "type": "static",
        "static": {"max_chunk_size_tokens": 512, "chunk_overlap_tokens": 128},
    },
)

print(f"File {file_url.id} loaded into Vector store OCP with ID: {vector_db_ocp_id}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/files "HTTP/1.1 200 OK"


Uploaded: file-ec5a87e24e904d6db90e3acab022797e


INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores/vs_35b4d3d7-6e9d-4adc-8d3f-686d9f62477b/files "HTTP/1.1 200 OK"


File file-ec5a87e24e904d6db90e3acab022797e loaded into Vector store OCP with ID: vs_35b4d3d7-6e9d-4adc-8d3f-686d9f62477b


In [33]:
vector_stores = client.vector_stores.list()

print(f"Vector stores created {vector_stores}")

INFO:httpx:HTTP Request: GET http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/vector_stores "HTTP/1.1 200 OK"


Vector stores created SyncOpenAICursorPage[VectorStore](data=[VectorStore(id='vs_2aba51eb-3c44-4d17-926e-136d358f8efa', created_at=1777587806, file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1), expires_after=None, expires_at=None, last_active_at=1777587806, metadata={'provider_id': 'milvus', 'provider_vector_store_id': 'vs_2aba51eb-3c44-4d17-926e-136d358f8efa', 'embedding_model': 'sentence-transformers/ibm-granite/granite-embedding-125m-english', 'embedding_dimension': '768'}, name='skus_rh_vector_db', object='vector_store', status='completed', usage_bytes=0), VectorStore(id='vs_35b4d3d7-6e9d-4adc-8d3f-686d9f62477b', created_at=1777587806, file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1), expires_after=None, expires_at=None, last_active_at=1777587806, metadata={'provider_id': 'milvus', 'provider_vector_store_id': 'vs_35b4d3d7-6e9d-4adc-8d3f-686d9f62477b', 'embedding_model': 'sentence-transformers/ibm-granite/granite-emb

In [34]:
query = "List of Red Hat OpenShift Container Platform SKU"

# Ask questions with file search
response = client.responses.create(
    model="vllm-inference/redhataillama-31-8b-instruct",
    input=query,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_db_skus_id],
    }],
)

logger.info(f"RAG Query from {vector_db_skus_name} - Result: \n{response.output_text}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/responses "HTTP/1.1 200 OK"
INFO:__main__:RAG Query from skus_rh_vector_db - Result: 
The Red Hat OpenShift Container Platform SKU includes various options such as:

* Red Hat OpenShift Container Platform (Bare Metal Node), Premium (1-2 Sockets up to 128 Cores)
* Red Hat OpenShift Container Platform (Bare Metal Node), Standard (1-2 Sockets up to 128 Cores)
* Red Hat OpenShift Container Platform (Bare Metal Node) Extended Update Support Long-Life Add-On - Term 1 (1-2 Sockets up to 128 Cores)
* Red Hat OpenShift Container Platform (Bare Metal Node) Extended Update Support Long-Life Add-On - Term 2 (1-2 Sockets up to 128 Cores)
* Red Hat OpenShift Container Platform Extended Update Support Long-Life Add-On - Term 1 (2 Cores or 4 vCPUs)
* Red Hat OpenShift Kubernetes Engine Extended Update Support Add-On for Distributed Computing (Edge Server) - Term 1 (1-2 Sockets up to 128 Core

In [35]:
query = "What is Red Hat OpenShift?"

# Ask questions with file search
response = client.responses.create(
    model="vllm-inference/redhataillama-31-8b-instruct",
    input=query,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_db_ocp_id],
    }],
)

logger.info(f"RAG Query from {vector_db_ocp_name} - Result: \n{response.output_text}")

INFO:httpx:HTTP Request: POST http://llama-stack-milvus-remote-service.proposal-rh-ai.svc.cluster.local:8321/v1/responses "HTTP/1.1 200 OK"
INFO:__main__:RAG Query from ocp_rh_vector_db - Result: 
Red Hat OpenShift is an enterprise-class platform built upon Kubernetes and provides a full DevOps product ready to use. It is designed with high availability and security in mind and integrates a whole host of DevOps tools in a single package. Among its features, we can find built-in user and group management, tighter security requirements for containers, more robust namespace isolation through projects, an integrated visual management console, an embedded container registry, a CI/CD pipeline system, a built-in Cloud Native application store featuring ready-to-use applications bundled as Kubernetes Operators, and integrated management and logging features. Red Hat OpenShift clusters feature more robust security defaults than stock Kubernetes and can also run in high-availability mode, ensuri